# SETTINGS

In [ ]:
import os
import numpy as np
import pandas as pd
import emoji

# CONFIGURATION

In [ ]:
project_dir = r"C:\cache\Youtube-ETL_Project"
raw_dir = os.path.join(project_dir, "data", "raw")
processed_dir = os.path.join(project_dir, "data", "processed")
latest_raw_path = os.path.join(raw_dir, "latest_raw_path.txt")

if os.path.exists(latest_raw_path):
    with open(latest_raw_path, "r", encoding="utf-8") as f:
        csv_files = f.read().strip()
else:
    raw_files = [
        os.path.join(raw_dir, file_name)
        for file_name in os.listdir(raw_dir)
        if file_name.startswith("raw_youtube_") and file_name.endswith(".csv")
    ]
    if not raw_files:
        raise FileNotFoundError(f"No raw CSV files found in: {raw_dir}")
    csv_files = max(raw_files, key=os.path.getmtime)

if not os.path.exists(csv_files):
    raise FileNotFoundError(f"Raw CSV not found: {csv_files}")

raw_file_name = os.path.basename(csv_files)
cleaned_file_name = raw_file_name.replace("raw_youtube_", "cleaned_youtube_", 1)

print("CSV file to process:", csv_files)

# READ CSV

In [ ]:
try:
    df = pd.read_csv(csv_files)
    print(f"Successfully read CSV files: {csv_files}, shape = {df.shape}")
    display(df.head(100))
except Exception as e:
    print(f"Error reading CSV files: {e}")
    raise

# OVERVIEW

In [ ]:
print("Shape:", df.shape)
print("\n--- Dtypes ---")
print(df.dtypes)
print("\n--- Info ---")
df.info()

In [ ]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
print("Các cột số:", numeric_cols)
display(df[numeric_cols].describe())

In [ ]:
# Data Pivot Table - Viewcount by Video ID and Category
df.pivot_table(index='video_title', columns='category', values='view_count', aggfunc=np.sum, fill_value=0)

In [ ]:
df.groupby("channel_name")["subscriber_count"].count().sort_values(ascending=False).head(10)

# CLEANING

In [ ]:
# Video title và tags: bỏ emoji (giữ nguyên logic bản cũ)
df["video_title"] = df["video_title"].fillna("").apply(lambda x: emoji.replace_emoji(x, replace="").strip())
df["tags"] = df["tags"].fillna("").apply(lambda x: emoji.replace_emoji(x, replace="").strip())
df

In [ ]:
# Duration categorization
df["duration_seconds"] = pd.to_timedelta(df["duration"]).dt.total_seconds().astype(int)

conditions1 = [
    df["duration_seconds"] >= 3600,
    df["duration_seconds"] >= 1800,
    df["duration_seconds"] >= 300,
]
choices1 = [
    "Long Video (1 hour or more)",
    "Medium Video (30-60 minutes)",
    "Short Video (5-30 minutes)",
]
df["duration_category"] = np.select(conditions1, choices1, default="Very Short Video (Less than 5 minutes)")
df

# RESULT

In [ ]:
cleaned_dir = processed_dir
os.makedirs(cleaned_dir, exist_ok=True)

cleaned_path = os.path.join(cleaned_dir, cleaned_file_name)
df.to_csv(cleaned_path, index=False, encoding="utf-8-sig")

latest_cleaned_path = os.path.join(cleaned_dir, "latest_cleaned_path.txt")
with open(latest_cleaned_path, "w", encoding="utf-8") as f:
    f.write(cleaned_path)

print(f"✅ Đã lưu: {cleaned_path} | shape = {df.shape}")
print(f"Latest cleaned path saved to: {latest_cleaned_path}")